## Unicode

**Unicode is a standard that assigns a unique numeric value, called a code point, to text symbols used by writing systems around the world.**

A universal dictionary that assigns a unique number to every character in every language. It gives every character from 'A' to '🫠'its own special ID number, called a code point.

Examples:

| Character | Unicode name | Code point |
|---|---|---|
| `A` | LATIN CAPITAL LETTER A | `U+0041` |
| `é` | LATIN SMALL LETTER E WITH ACUTE | `U+00E9` |
| `€` | EURO SIGN | `U+20AC` |
| `🐍` | SNAKE | `U+1F40D` |

Unicode answers this question:

> Which abstract symbol does this number represent?


In [24]:
# ord(character)
# Converts one Unicode character into its integer Unicode code point.
print(ord("A"))
print(ord("a"))

print(ord("$"))
print(ord("€"))

# print(ord('Hello World'))  # This will raise an error because ord() expects a single character.


# chr(number)
# Performs the reverse operation: converts a Unicode code point into its character.

print(chr(65))
print(chr(97))
print(chr(8364))  # '€'
print(chr(0x1F600))  # '😀' 

# Python can hold this as text:
text = "é"
# Internally, Python understands it as Unicode. But a file needs actual byte values. So Python must encode it:
encoded = text.encode("utf-8")
print(encoded) 

65
97
36
8364
A
a
€
😀
b'\xc3\xa9'


Unicode does **not** by itself specify the bytes stored in a file. An encoding such as UTF-8 converts Unicode code points into bytes.

In [ ]:
characters = ["A", "é", "€", "🐍"]

for character in characters:
    code_point = ord(character)
    print(
        repr(character),
        "-> integer:", code_point,
        "-> Unicode:", f"U+{code_point:04X}",
        "-> back again:", chr(code_point),
    )

'A' -> integer: 65 -> Unicode: U+0041 -> back again: A
'é' -> integer: 233 -> Unicode: U+00E9 -> back again: é
'€' -> integer: 8364 -> Unicode: U+20AC -> back again: €
'🐍' -> integer: 128013 -> Unicode: U+1F40D -> back again: 🐍


## Character encoding: Unicode bytes

Unicode assigns a numeric code point to each character.
Examples:

'A' → U+0041 → 65
'€' → U+20AC → 8364
'你' → U+4F60 → 20320

These numbers identify characters, but computers ultimately store and transmit data as bytes.

Unicode tells us which number identifies a character, but it does not by itself specify the exact byte sequence that should be written to a file or sent over a network.

We need a shared set of rules that determines:

how many bytes represent a code point,
which byte values should be used,
how the bytes should be ordered,
where one encoded character ends and the next begins,
which byte sequences are valid or invalid.

These rules are called a character encoding.

#### Encoding
An encoding is a set of rules for converting Unicode code points into byte sequences and converting those bytes back into Unicode text.

There are three main ways here to store these IDs. 

- **UTF-32:** uses 4 bytes for every code point.
- **UTF-16:** uses 2 bytes for code points in the Basic Multilingual Plane and 4 bytes, represented by a surrogate pair, for supplementary code points.
- **UTF-8:** uses a variable-width sequence of 1 to 4 bytes. ASCII characters need 1 byte, while other code points expand to 2, 3, or 4 bytes when necessary.

UTF stands for Unicode Transformation Format

UTF-8, UTF-16, and UTF-32 encode the same Unicode code points using different byte representations.
UTF-8 is the dominant encoding on the web. It is compact for ASCII text and can represent every Unicode code point.


| Encoding | Rule | `A` (65) | `€` (8364) |
|---|---|---:|---:|
| UTF-32 BE | Always use 4 bytes | `[0, 0, 0, 65]` | `[0, 0, 32, 172]` |
| UTF-16 BE | Use 2 bytes here; some code points need 4 | `[0, 65]` | `[32, 172]` |
| UTF-8 | Use 1–4 bytes as needed | `[65]` | `[226, 130, 172]` |

For ASCII text such as `Hello`, UTF-8 uses only 5 bytes. UTF-16 uses 10 bytes, and UTF-32 uses 20 bytes when byte-order marks are excluded. 


### UTF-8

UTF-8 is one method for encoding Unicode code points as bytes.

UTF-8 uses a variable number of bytes:

```text
Code-point range	UTF-8 size
U+0000 to U+007F	1 byte
U+0080 to U+07FF	2 bytes
U+0800 to U+FFFF	3 bytes
U+10000 to U+10FFFF	4 bytes
```

Examples:

```text
'A'  → 1 UTF-8 byte
'é'  → 2 UTF-8 bytes
'€'  → 3 UTF-8 bytes
'你' → 3 UTF-8 bytes
'😀' → 4 UTF-8 bytes
```

UTF-8 uses recognizable bit prefixes in the leading byte:

```text
0xxxxxxx  → a complete 1-byte character [standard ASCII]
110xxxxx  → the start of a 2-byte sequence
1110xxxx  → the start of a 3-byte sequence
11110xxx  → the start of a 4-byte sequence
10xxxxxx  → a continuation byte
```

The `x` bits carry pieces of the code-point value. Continuation bytes let a decoder recognize which bytes belong to the same sequence.

Example : 
For a three-byte character 1110xxxx 10xxxxxx 10xxxxxx

For  `€`:
11100010 10000010 10101100

The prefixes have specific meanings:

1110xxxx → beginning of a three-byte sequence
10xxxxxx → continuation byte
10xxxxxx → continuation byte

These patterns allow the decoder to determine how many bytes belong to the character, where the character begins,where the next character begins.



In [18]:
# Encoding
# Encoding converts Unicode text into bytes.

text = "€"

data = text.encode("utf-8")

print(data)
print(list(data))  # Show the individual byte values in the encoded data.
print(type(data))  

# Decoding
# Decoding converts bytes back into Unicode text.

data = b'\xe2\x82\xac'

text = data.decode("utf-8")

print(text)
print(type(text))  

b'\xe2\x82\xac'
[226, 130, 172]
<class 'bytes'>
€
<class 'str'>


Example: UTF-8 Encoding of `漢`

```text
Character:          漢
Unicode code point: U+6F22
Decimal:            28450
Binary:             110111100100010
```

The binary representation of `U+6F22` is:

```text
110111100100010
```

It contains **15 bits**.

UTF-8 provides the following patterns:

```text
1 byte:  0xxxxxxx
2 bytes: 110xxxxx 10xxxxxx
3 bytes: 1110xxxx 10xxxxxx 10xxxxxx
4 bytes: 11110xxx 10xxxxxx 10xxxxxx 10xxxxxx
```

Because the code point requires only 15 bits, it fits inside the three-byte pattern:

```text
1110xxxx 10xxxxxx 10xxxxxx
```

Therefore, `漢` requires **three UTF-8 bytes**

The three-byte pattern contains 16 `x` positions.

The code point contains 15 bits:

```text
110111100100010
```

Add one leading zero to produce 16 bits:

```text
0110111100100010
```

The three-byte pattern divides its data positions into groups of:

```text
4 bits | 6 bits | 6 bits
```

Split the padded 16-bit value from left to right:

```text
0110 | 111100 | 100010
```

These groups come directly from the available `x` positions:

```text
1110xxxx | 10xxxxxx | 10xxxxxx
     4   |     6    |     6
```

```text
First group:  0110
Second group: 111100
Third group:  100010
```

```text
1110xxxx 10xxxxxx 10xxxxxx
```

Insert the three groups:

```text
1110 0110
10 111100
10 100010
```

```text
11100110 10111100 10100010
```

Final UTF-8 Representation

```text
Binary:      11100110 10111100 10100010
Hexadecimal: E6 BC A2
Decimal:     230, 188, 162
```

> UTF-8 prefix bits describe the structure of the encoded sequence. The `x` positions carry the actual bits of the Unicode code point.


In [25]:
print("漢".encode("utf-8"))
print(list("漢".encode("utf-8")))


b'\xe6\xbc\xa2'
[230, 188, 162]


In [6]:
samples = ["A", "é", "€", "🐍", "hello", "アグニー"]

for text in samples:
    utf8_bytes = text.encode("utf-8")
    print(f"Text: {text!r}")
    print(f"  Python characters: {len(text)}")
    print(f"  UTF-8 bytes:        {len(utf8_bytes)}")
    print(f"  Raw bytes:          {utf8_bytes}")
    print(f"  Byte integers:      {list(utf8_bytes)}")
    print()

Text: 'A'
  Python characters: 1
  UTF-8 bytes:        1
  Raw bytes:          b'A'
  Byte integers:      [65]

Text: 'é'
  Python characters: 1
  UTF-8 bytes:        2
  Raw bytes:          b'\xc3\xa9'
  Byte integers:      [195, 169]

Text: '€'
  Python characters: 1
  UTF-8 bytes:        3
  Raw bytes:          b'\xe2\x82\xac'
  Byte integers:      [226, 130, 172]

Text: '🐍'
  Python characters: 1
  UTF-8 bytes:        4
  Raw bytes:          b'\xf0\x9f\x90\x8d'
  Byte integers:      [240, 159, 144, 141]

Text: 'hello'
  Python characters: 5
  UTF-8 bytes:        5
  Raw bytes:          b'hello'
  Byte integers:      [104, 101, 108, 108, 111]

Text: 'アグニー'
  Python characters: 4
  UTF-8 bytes:        12
  Raw bytes:          b'\xe3\x82\xa2\xe3\x82\xb0\xe3\x83\x8b\xe3\x83\xbc'
  Byte integers:      [227, 130, 162, 227, 130, 176, 227, 131, 139, 227, 131, 188]



Encoding and decoding should round-trip:

```text
Unicode text --encode--> UTF-8 bytes --decode--> Unicode text
```

In [7]:
original_text = "BPE learns from text 🤖"
encoded_bytes = original_text.encode("utf-8")
decoded_text = encoded_bytes.decode("utf-8")

print("Original:", original_text)
print("Encoded: ", encoded_bytes)
print("Decoded: ", decoded_text)

assert decoded_text == original_text

Original: BPE learns from text 🤖
Encoded:  b'BPE learns from text \xf0\x9f\xa4\x96'
Decoded:  BPE learns from text 🤖


Inspecting Unicode names

Python's standard-library `unicodedata` module lets us inspect Unicode properties.

In [14]:
import unicodedata

text = "Bé! 🐍"

for character in text:
    name = unicodedata.name(character, "NO NAME")
    category = unicodedata.category(character)
    print(
        f"{character!r:5}",
        f"{f'U+{ord(character):04X}':10}",
        f"category={category:3}",
        name,
    )

'B'   U+0042     category=Lu  LATIN CAPITAL LETTER B
'é'   U+00E9     category=Ll  LATIN SMALL LETTER E WITH ACUTE
'!'   U+0021     category=Po  EXCLAMATION MARK
' '   U+0020     category=Zs  SPACE
'🐍'   U+1F40D    category=So  SNAKE


#### Byte Order

**Byte order** describes how the bytes of a multi-byte value are arranged.

Example:

```text
Value: 0x1234
Bytes: 12 34
```

Big-endian
Stores the most significant byte first:

```text
12 34
```

Little-endian
Stores the least significant byte first:

```text
34 12
```

Both represent the same value, but the reader must use the correct byte order.


Byte order in Unicode encodings

UTF-16

Each code unit uses 2 bytes.

For `A` (`U+0041`):

```text
UTF-16BE: 00 41
UTF-16LE: 41 00
```

UTF-32

Each code unit uses 4 bytes.

For `A`:

```text
UTF-32BE: 00 00 00 41
UTF-32LE: 41 00 00 00
```

UTF-8

UTF-8 has no endianness because it uses individual bytes in a fixed sequence.

```text
€ → E2 82 AC
```


---

#### Byte Order Mark

A **Byte Order Mark (BOM)** may appear at the start of a file to indicate encoding or byte order.

```text
UTF-16BE: FE FF
UTF-16LE: FF FE
UTF-8:    EF BB BF
```

The UTF-8 BOM is optional and does not indicate endianness.

#### SUMMARY 
Unicode : Unicode assigns a numeric code point to each character.
€ → U+20AC

UTF-8 : UTF-8 determines how that code point is represented as bytes.
U+20AC → E2 82 AC

Therefore:

Character → Unicode code point → encoded bytes

Example:

```text
€                           Character
U+20AC                      Unicode code point
8364                        Decimal representation
E2 82 AC                    UTF-8 bytes
11100010 10000010 10101100  Stored binary bits
```

In [21]:
text = "café ☕"

code_point_units = [character for character in text]
byte_units = list(text.encode("utf-8"))

print("Original text:    ", text)
print("Code-point units:", code_point_units)
print("Byte units:      ", byte_units)
print("Decoded bytes:   ", bytes(byte_units).decode("utf-8"))

Original text:     café ☕
Code-point units: ['c', 'a', 'f', 'é', ' ', '☕']
Byte units:       [99, 97, 102, 195, 169, 32, 226, 152, 149]
Decoded bytes:    café ☕
